# Cell Tuning Analysis

This notebook implements comprehensive statistical tests to determine if cells are tuned to:
1. **Task modulation** (Baseline vs GO)
2. **Direction tuning** (Left vs Right)
3. **Signal processing** (STOP vs CONT, **correctly aligned to stop_cue**)
4. **Signal vs GO** (Does STOP/CONT signal change activity differently than GO?)

**IMPORTANT**: STOP/CONT trials are aligned to `stop_cue` (not `go_cue`) to properly measure signal responses.

**Methods:**
- ANOVA for main effects
- Effect size measures (Cohen's d, eta-squared)
- Direction and signal selectivity indices
- Multiple comparison correction (Bonferroni)

In [46]:
# Imports
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import holoviews as hv
import hvplot.pandas
hv.extension('bokeh')

from scipy import stats
from scipy.stats import f_oneway, ttest_ind
from pprint import pprint
from itertools import combinations

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

# Import classes
import importlib
import session_class
import cell_analysis
importlib.reload(session_class)
importlib.reload(cell_analysis)
from session_class import Session
from cell_analysis import Cell, PopulationAnalyzer

print("✓ Imports loaded successfully!")

✓ Imports loaded successfully!


## 1. Load Data

In [55]:
# Load MSN cell database
monkey = 'fiona'
base_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data' 
pickle_file = base_path / f'msn_{monkey}_cell_trial_data.pkl'
cell_df = pd.read_pickle(pickle_file)

# Select session with most cells
session_cell_counts = cell_df.groupby('trial_session')['cell_ID'].nunique().sort_values(ascending=False)
best_session_id = session_cell_counts.index[0]
best_session_id = "fi210810a"

print(f"Using session: {best_session_id}")
print(f"Number of cells: {session_cell_counts.iloc[0]}")

# Create Session object
session_data = cell_df[cell_df['trial_session'] == best_session_id]
session = Session(session_data, verbose=True)

Using session: fi210810a
Number of cells: 85
Session fi210810a initialized:
  - Number of cells: 7
  - Total trials: 510
  - Trial types: ['CONT', 'GO', 'STOP']
  - Directions: [np.int64(0), np.int64(180)]


## 2. Helper Functions - CORRECTED

In [56]:
def get_firing_rates(cell, align_event, window, trial_type=None, direction=None, success_only=True):
    """
    Extract firing rates for specific conditions.
    
    Parameters:
    -----------
    cell : Cell
        Cell object
    align_event : str
        'go_cue' or 'stop_cue'
    window : list
        Time window [start, end] in ms
    trial_type : str, optional
        'GO', 'STOP', or 'CONT'
    direction : int, optional
        0 (right) or 180 (left)
    success_only : bool
        Include only successful trials
    
    Returns:
    --------
    np.array : Firing rates in spikes/sec
    """
    # Filter trials
    df = cell.filter_trials(trial_type=trial_type, direction=direction, success_only=success_only)
    
    rates = []
    for _, row in df.iterrows():
        # Get alignment time
        if align_event == 'go_cue':
            t0 = row['go_cue']
        elif align_event == 'stop_cue':
            t0 = row['stop_cue']
        else:
            raise ValueError(f"Unknown align_event: {align_event}")
        
        # Skip if alignment event is missing
        if pd.isna(t0):
            continue
            
        spikes = np.array(row['neural_data'], dtype=float)
        aligned_spikes = spikes - t0
        
        # Count spikes in window
        count = np.sum((aligned_spikes >= window[0]) & (aligned_spikes <= window[1]))
        duration = (window[1] - window[0]) / 1000.0  # Convert to seconds
        rates.append(count / duration)
        
    return np.array(rates)


def cohens_d(group1, group2):
    """Calculate Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    if n1 < 2 or n2 < 2:
        return np.nan
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    if pooled_std == 0:
        return 0.0
    
    return (np.mean(group1) - np.mean(group2)) / pooled_std


def eta_squared(groups):
    """Calculate eta-squared effect size for ANOVA."""
    all_data = np.concatenate(groups)
    grand_mean = np.mean(all_data)
    
    ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
    ss_total = np.sum((all_data - grand_mean)**2)
    
    if ss_total == 0:
        return 0.0
    
    return ss_between / ss_total


def direction_selectivity_index(left_rate, right_rate):
    """Calculate direction selectivity index."""
    if left_rate + right_rate == 0:
        return 0.0
    return (left_rate - right_rate) / (left_rate + right_rate)


def modulation_index(baseline_rate, task_rate):
    """Calculate modulation index."""
    if baseline_rate + task_rate == 0:
        return 0.0
    return (task_rate - baseline_rate) / (task_rate + baseline_rate)


def signal_selectivity_index(stop_rate, cont_rate):
    """Calculate signal selectivity index (STOP vs CONT)."""
    if stop_rate + cont_rate == 0:
        return 0.0
    return (stop_rate - cont_rate) / (stop_rate + cont_rate)

print("✓ Helper functions defined")

✓ Helper functions defined


## 3. Main Tuning Analysis Function - CORRECTED

In [57]:
def analyze_cell_tuning(cell, alpha=0.05, verbose=False):
    """
    Comprehensive tuning analysis with CORRECTED alignment for STOP/CONT trials.
    
    Key fix: STOP and CONT trials are aligned to stop_cue, not go_cue.
    
    Parameters:
    -----------
    cell : Cell
        Cell object to analyze
    alpha : float
        Significance level (default: 0.05)
    verbose : bool
        Print detailed results
    
    Returns:
    --------
    dict : Comprehensive tuning results
    """
    results = {
        'cell_id': cell.cell_id,
        'cell_type': cell.cell_type,
    }
    
    n_tests = 6  # Increased: task_mod, direction, stop_mod, cont_mod, signal_discrimination, go_vs_signal
    bonferroni_alpha = alpha / n_tests
    
    # ========================================
    # 1. BASELINE (from GO trials)
    # ========================================
    baseline_rates = get_firing_rates(cell, 'go_cue', [-500, 0], trial_type='GO')
    
    # ========================================
    # 2. TASK MODULATION (Baseline vs GO)
    # ========================================
    go_rates = get_firing_rates(cell, 'go_cue', [0, 200], trial_type='GO')
    
    if len(baseline_rates) > 0 and len(go_rates) > 0:
        f_stat, p_val = f_oneway(baseline_rates, go_rates)
        effect_size = cohens_d(go_rates, baseline_rates)
        eta2 = eta_squared([baseline_rates, go_rates])
        
        results['task_modulation'] = {
            'F': f_stat,
            'p': p_val,
            'p_bonferroni': p_val * n_tests,
            'significant': p_val < bonferroni_alpha,
            'cohens_d': effect_size,
            'eta_squared': eta2,
            'modulation_index': modulation_index(np.mean(baseline_rates), np.mean(go_rates)),
            'baseline_mean': np.mean(baseline_rates),
            'go_mean': np.mean(go_rates)
        }
    else:
        results['task_modulation'] = {'significant': False}
    
    # ========================================
    # 3. DIRECTION TUNING (on GO trials)
    # ========================================
    go_left = get_firing_rates(cell, 'go_cue', [0, 200], trial_type='GO', direction=180)
    go_right = get_firing_rates(cell, 'go_cue', [0, 200], trial_type='GO', direction=0)
    
    if len(go_left) > 0 and len(go_right) > 0:
        f_stat, p_val = f_oneway(go_left, go_right)
        effect_size = cohens_d(go_left, go_right)
        dsi = direction_selectivity_index(np.mean(go_left), np.mean(go_right))
        
        results['direction_tuning'] = {
            'F': f_stat,
            'p': p_val,
            'p_bonferroni': p_val * n_tests,
            'significant': p_val < bonferroni_alpha,
            'cohens_d': effect_size,
            'direction_selectivity_index': dsi,
            'preferred_direction': 'left' if dsi > 0 else 'right',
            'left_mean': np.mean(go_left),
            'right_mean': np.mean(go_right)
        }
    else:
        results['direction_tuning'] = {'significant': False}
    
    # ========================================
    # 4. STOP MODULATION (Baseline vs STOP response)
    # CORRECTED: Align to stop_cue!
    # ========================================
    stop_rates = get_firing_rates(cell, 'stop_cue', [0, 200], trial_type='STOP')
    
    if len(baseline_rates) > 0 and len(stop_rates) > 0:
        f_stat, p_val = f_oneway(baseline_rates, stop_rates)
        effect_size = cohens_d(stop_rates, baseline_rates)
        mi = modulation_index(np.mean(baseline_rates), np.mean(stop_rates))
        
        results['stop_modulation'] = {
            'F': f_stat,
            'p': p_val,
            'p_bonferroni': p_val * n_tests,
            'significant': p_val < bonferroni_alpha,
            'cohens_d': effect_size,
            'modulation_index': mi,
            'stop_mean': np.mean(stop_rates)
        }
    else:
        results['stop_modulation'] = {'significant': False}
    
    # ========================================
    # 5. CONT MODULATION (Baseline vs CONT response)
    # CORRECTED: Align to stop_cue!
    # ========================================
    cont_rates = get_firing_rates(cell, 'stop_cue', [0, 200], trial_type='CONT')
    
    if len(baseline_rates) > 0 and len(cont_rates) > 0:
        f_stat, p_val = f_oneway(baseline_rates, cont_rates)
        effect_size = cohens_d(cont_rates, baseline_rates)
        mi = modulation_index(np.mean(baseline_rates), np.mean(cont_rates))
        
        results['cont_modulation'] = {
            'F': f_stat,
            'p': p_val,
            'p_bonferroni': p_val * n_tests,
            'significant': p_val < bonferroni_alpha,
            'cohens_d': effect_size,
            'modulation_index': mi,
            'cont_mean': np.mean(cont_rates)
        }
    else:
        results['cont_modulation'] = {'significant': False}
    
    # ========================================
    # 6. SIGNAL DISCRIMINATION (STOP vs CONT)
    # Both aligned to stop_cue
    # ========================================
    if len(stop_rates) > 0 and len(cont_rates) > 0:
        f_stat, p_val = f_oneway(stop_rates, cont_rates)
        effect_size = cohens_d(stop_rates, cont_rates)
        ssi = signal_selectivity_index(np.mean(stop_rates), np.mean(cont_rates))
        
        results['signal_discrimination'] = {
            'F': f_stat,
            'p': p_val,
            'p_bonferroni': p_val * n_tests,
            'significant': p_val < bonferroni_alpha,
            'cohens_d': effect_size,
            'signal_selectivity_index': ssi,
            'preferred_signal': 'STOP' if ssi > 0 else 'CONT'
        }
    else:
        results['signal_discrimination'] = {'significant': False}
    
    # ========================================
    # 7. GO vs SIGNALS (does signal differ from GO?)
    # Compare GO to combined STOP+CONT
    # ========================================
    if len(go_rates) > 0 and len(stop_rates) > 0 and len(cont_rates) > 0:
        # Test GO vs STOP
        f_stat_stop, p_val_stop = f_oneway(go_rates, stop_rates)
        d_stop = cohens_d(go_rates, stop_rates)
        
        # Test GO vs CONT
        f_stat_cont, p_val_cont = f_oneway(go_rates, cont_rates)
        d_cont = cohens_d(go_rates, cont_rates)
        
        # Overall test: GO vs STOP vs CONT
        f_stat_all, p_val_all = f_oneway(go_rates, stop_rates, cont_rates)
        
        results['go_vs_signals'] = {
            'F_overall': f_stat_all,
            'p_overall': p_val_all,
            'p_bonferroni': p_val_all * n_tests,
            'significant': p_val_all < bonferroni_alpha,
            'go_vs_stop_p': p_val_stop,
            'go_vs_stop_d': d_stop,
            'go_vs_cont_p': p_val_cont,
            'go_vs_cont_d': d_cont
        }
    else:
        results['go_vs_signals'] = {'significant': False}
    
    # ========================================
    # SUMMARY CLASSIFICATION
    # ========================================
    results['summary'] = {
        'is_task_modulated': results['task_modulation'].get('significant', False),
        'is_direction_tuned': results['direction_tuning'].get('significant', False),
        'is_stop_modulated': results['stop_modulation'].get('significant', False),
        'is_cont_modulated': results['cont_modulation'].get('significant', False),
        'is_signal_discriminative': results['signal_discrimination'].get('significant', False),
        'is_go_vs_signal_different': results['go_vs_signals'].get('significant', False),
    }
    
    # Build tuning profile
    profile = []
    if results['summary']['is_task_modulated']:
        profile.append('task_modulated')
    if results['summary']['is_direction_tuned']:
        profile.append(f"direction_tuned_{results['direction_tuning'].get('preferred_direction', 'none')}")
    if results['summary']['is_stop_modulated']:
        profile.append('stop_modulated')
    if results['summary']['is_cont_modulated']:
        profile.append('cont_modulated')
    if results['summary']['is_signal_discriminative']:
        profile.append(f"signal_selective_{results['signal_discrimination'].get('preferred_signal', 'none')}")
    if results['summary']['is_go_vs_signal_different']:
        profile.append('go_vs_signal_different')
    
    results['summary']['tuning_profile'] = profile if profile else ['not_tuned']
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"Cell {cell.cell_id} Tuning Analysis (CORRECTED)")
        print(f"{'='*70}")
        print(f"\nProfile: {', '.join(results['summary']['tuning_profile'])}")
        
        print(f"\n{'Task Modulation':<25s}: {'YES' if results['summary']['is_task_modulated'] else 'NO'}")
        if results['task_modulation'].get('significant'):
            print(f"  - Baseline: {results['task_modulation']['baseline_mean']:.2f} sp/s")
            print(f"  - GO:       {results['task_modulation']['go_mean']:.2f} sp/s")
            print(f"  - p = {results['task_modulation']['p']:.4e}")
        
        print(f"\n{'Direction Tuning':<25s}: {'YES' if results['summary']['is_direction_tuned'] else 'NO'}")
        if results['direction_tuning'].get('significant'):
            print(f"  - Preferred: {results['direction_tuning']['preferred_direction']}")
            print(f"  - p = {results['direction_tuning']['p']:.4e}")
        
        print(f"\n{'STOP Modulation':<25s}: {'YES' if results['summary']['is_stop_modulated'] else 'NO'}")
        if results['stop_modulation'].get('significant'):
            print(f"  - STOP: {results['stop_modulation']['stop_mean']:.2f} sp/s")
            print(f"  - p = {results['stop_modulation']['p']:.4e}")
        
        print(f"\n{'CONT Modulation':<25s}: {'YES' if results['summary']['is_cont_modulated'] else 'NO'}")
        if results['cont_modulation'].get('significant'):
            print(f"  - CONT: {results['cont_modulation']['cont_mean']:.2f} sp/s")
            print(f"  - p = {results['cont_modulation']['p']:.4e}")
        
        print(f"\n{'Signal Discrimination':<25s}: {'YES' if results['summary']['is_signal_discriminative'] else 'NO'}")
        if results['signal_discrimination'].get('significant'):
            print(f"  - Preferred: {results['signal_discrimination']['preferred_signal']}")
            print(f"  - p = {results['signal_discrimination']['p']:.4e}")
        
        print(f"\n{'GO vs Signals':<25s}: {'YES' if results['summary']['is_go_vs_signal_different'] else 'NO'}")
        if results['go_vs_signals'].get('significant'):
            print(f"  - p = {results['go_vs_signals']['p_overall']:.4e}")
    
    return results

print("✓ Main analysis function defined (CORRECTED)")

✓ Main analysis function defined (CORRECTED)


## 4. Test on Single Cell

In [58]:
# Test on a single cell
cell_id = session.cell_ids[0]
cell = session.get_cell(cell_id)

print(f"Analyzing: {cell}\n")

# Run analysis
results = analyze_cell_tuning(cell, verbose=True)

Analyzing: Cell 9455 | Type: msn | Trials: 510 | Directions: [np.int64(0), np.int64(180)] | Trial Types: ['CONT', 'GO', 'STOP'] | SSDs: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]


Cell 9455 Tuning Analysis (CORRECTED)

Profile: cont_modulated

Task Modulation          : NO

Direction Tuning         : NO

STOP Modulation          : NO

CONT Modulation          : YES
  - CONT: 1.76 sp/s
  - p = 3.0939e-04

Signal Discrimination    : NO

GO vs Signals            : NO


## 5. Analyze All Cells

In [59]:
# Analyze all cells
all_results = []

print(f"Analyzing {len(session.cell_ids)} cells...\n")

for i, cell_id in enumerate(session.cell_ids):
    if (i + 1) % 10 == 0:
        print(f"  Processed {i + 1}/{len(session.cell_ids)} cells")
    
    cell = session.get_cell(cell_id)
    result = analyze_cell_tuning(cell, verbose=False)
    all_results.append(result)

print(f"\n✓ Analysis complete for {len(all_results)} cells")

Analyzing 7 cells...


✓ Analysis complete for 7 cells


## 6. Summary Statistics - CORRECTED

In [60]:
# Create summary DataFrame
summary_data = []

for res in all_results:
    summary_data.append({
        'cell_id': res['cell_id'],
        'is_task_modulated': res['summary']['is_task_modulated'],
        'is_direction_tuned': res['summary']['is_direction_tuned'],
        'is_stop_modulated': res['summary']['is_stop_modulated'],
        'is_cont_modulated': res['summary']['is_cont_modulated'],
        'is_signal_discriminative': res['summary']['is_signal_discriminative'],
        'is_go_vs_signal_different': res['summary']['is_go_vs_signal_different'],
        'preferred_direction': res['direction_tuning'].get('preferred_direction', 'none'),
        'preferred_signal': res['signal_discrimination'].get('preferred_signal', 'none'),
        'tuning_profile': ', '.join(res['summary']['tuning_profile']),
        'task_mod_p': res['task_modulation'].get('p', np.nan),
        'dir_tuning_p': res['direction_tuning'].get('p', np.nan),
        'stop_mod_p': res['stop_modulation'].get('p', np.nan),
        'cont_mod_p': res['cont_modulation'].get('p', np.nan),
        'signal_disc_p': res['signal_discrimination'].get('p', np.nan),
        'go_vs_signal_p': res['go_vs_signals'].get('p_overall', np.nan),
        'baseline_FR': res['task_modulation'].get('baseline_mean', np.nan),
        'go_FR': res['task_modulation'].get('go_mean', np.nan),
        'stop_FR': res['stop_modulation'].get('stop_mean', np.nan),
        'cont_FR': res['cont_modulation'].get('cont_mean', np.nan),
        'DSI': res['direction_tuning'].get('direction_selectivity_index', np.nan),
        'SSI': res['signal_discrimination'].get('signal_selectivity_index', np.nan),
    })

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*70)
print("POPULATION SUMMARY (CORRECTED ANALYSIS)")
print("="*70)
print(f"\nTotal cells analyzed: {len(summary_df)}")
print(f"\nTuning properties:")
print(f"  - Task modulated (GO):        {summary_df['is_task_modulated'].sum():3d} ({summary_df['is_task_modulated'].sum()/len(summary_df)*100:5.1f}%)")
print(f"  - Direction tuned:            {summary_df['is_direction_tuned'].sum():3d} ({summary_df['is_direction_tuned'].sum()/len(summary_df)*100:5.1f}%)")
print(f"  - STOP modulated:             {summary_df['is_stop_modulated'].sum():3d} ({summary_df['is_stop_modulated'].sum()/len(summary_df)*100:5.1f}%)")
print(f"  - CONT modulated:             {summary_df['is_cont_modulated'].sum():3d} ({summary_df['is_cont_modulated'].sum()/len(summary_df)*100:5.1f}%)")
print(f"  - Signal discriminative:      {summary_df['is_signal_discriminative'].sum():3d} ({summary_df['is_signal_discriminative'].sum()/len(summary_df)*100:5.1f}%)")
print(f"  - GO vs Signal different:     {summary_df['is_go_vs_signal_different'].sum():3d} ({summary_df['is_go_vs_signal_different'].sum()/len(summary_df)*100:5.1f}%)")

print(f"\nSignal preferences (among signal-discriminative cells):")
sig_cells = summary_df[summary_df['is_signal_discriminative']]
if len(sig_cells) > 0:
    sig_counts = sig_cells['preferred_signal'].value_counts()
    for signal, count in sig_counts.items():
        print(f"  - {signal}: {count}")
else:
    print(f"  - None")

# Display summary table
print("\n" + "="*70)
print("Sample of results:")
print("="*70)
display(summary_df[['cell_id', 'is_task_modulated', 'is_direction_tuned', 
                     'is_stop_modulated', 'is_cont_modulated', 
                     'is_signal_discriminative', 'tuning_profile']].head(20))


POPULATION SUMMARY (CORRECTED ANALYSIS)

Total cells analyzed: 7

Tuning properties:
  - Task modulated (GO):          3 ( 42.9%)
  - Direction tuned:              2 ( 28.6%)
  - STOP modulated:               4 ( 57.1%)
  - CONT modulated:               6 ( 85.7%)
  - Signal discriminative:        0 (  0.0%)
  - GO vs Signal different:       2 ( 28.6%)

Signal preferences (among signal-discriminative cells):
  - None

Sample of results:


,cell_id,is_task_modulated,is_direction_tuned,is_stop_modulated,is_cont_modulated,is_signal_discriminative,tuning_profile
0,9455,False,False,False,True,False,cont_modulated
1,9458,False,False,False,False,False,not_tuned
2,9460,True,False,True,True,False,"task_modulated, stop_modulated, cont_modulated"
3,9461,False,True,False,True,False,"direction_tuned_left, cont_modulated"
4,9462,False,False,True,True,False,"stop_modulated, cont_modulated, go_vs_signal_d..."
5,9464,True,True,True,True,False,"task_modulated, direction_tuned_right, stop_mo..."
6,9467,True,False,True,True,False,"task_modulated, stop_modulated, cont_modulated"


## 7. Comparison with Old Analysis

In [61]:
print("\n" + "="*70)
print("KEY DIFFERENCES FROM OLD ANALYSIS")
print("="*70)
print("""
OLD (INCORRECT) APPROACH:
  - STOP/CONT trials aligned to go_cue
  - Measured activity [0-200ms] after GO cue
  - Problem: This captures the GO response, not STOP/CONT signal response!

NEW (CORRECTED) APPROACH:
  - STOP/CONT trials aligned to stop_cue
  - Measured activity [0-200ms] after STOP/CONT signal
  - Correctly captures the cell's response to the actual signal

NEW MEASURES:
  1. STOP modulation: Does cell respond to STOP signal?
  2. CONT modulation: Does cell respond to CONT signal?
  3. Signal discrimination: Can cell distinguish STOP from CONT?
  4. GO vs Signals: Is signal response different from GO response?
""")

print("\nExpected changes:")
print("  - May see MORE signal-discriminative cells (now properly aligned)")
print("  - STOP/CONT modulation now meaningful (was confounded with GO before)")
print("  - Can now test if signals modulate activity differently than GO")


KEY DIFFERENCES FROM OLD ANALYSIS

OLD (INCORRECT) APPROACH:
  - STOP/CONT trials aligned to go_cue
  - Measured activity [0-200ms] after GO cue
  - Problem: This captures the GO response, not STOP/CONT signal response!

NEW (CORRECTED) APPROACH:
  - STOP/CONT trials aligned to stop_cue
  - Measured activity [0-200ms] after STOP/CONT signal
  - Correctly captures the cell's response to the actual signal

NEW MEASURES:
  1. STOP modulation: Does cell respond to STOP signal?
  2. CONT modulation: Does cell respond to CONT signal?
  3. Signal discrimination: Can cell distinguish STOP from CONT?
  4. GO vs Signals: Is signal response different from GO response?


Expected changes:
  - May see MORE signal-discriminative cells (now properly aligned)
  - STOP/CONT modulation now meaningful (was confounded with GO before)
  - Can now test if signals modulate activity differently than GO


## 8. Save Results

In [62]:
# Create output directory
output_path = Path.cwd().parent / 'data' / 'tuning_analysis_v2'
output_path.mkdir(exist_ok=True)

# Save summary DataFrame
csv_file = output_path / f'tuning_summary_corrected_{best_session_id}.csv'
summary_df.to_csv(csv_file, index=False)
print(f"✓ Saved summary to: {csv_file}")

# Save full results as pickle
pkl_file = output_path / f'tuning_full_results_corrected_{best_session_id}.pkl'
pd.to_pickle(all_results, pkl_file)
print(f"✓ Saved full results to: {pkl_file}")

✓ Saved summary to: /Users/barak/Projects/population_analysis/data/tuning_analysis_v2/tuning_summary_corrected_fi210810a.csv
✓ Saved full results to: /Users/barak/Projects/population_analysis/data/tuning_analysis_v2/tuning_full_results_corrected_fi210810a.pkl
